# Notebook 7: Discriminative vs Generative Models

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2 hours | **Theme:** Spam Email Detection

---

## Why This Matters

Every ML classification algorithm belongs to one of two philosophical camps:

- **Discriminative models** ask: *"Given this email, is it spam?"* — they learn the boundary between classes.
- **Generative models** ask: *"What does a typical spam email look like?"* — they model each class and use Bayes' theorem to classify.

Understanding this distinction lets you choose the right tool:
- Got lots of labeled data? Discriminative models usually win.
- Need to generate synthetic data, handle missing features, or detect anomalies? Generative models shine.
- Got 10,000 unlabeled + 100 labeled emails? Only generative models can exploit the unlabeled data.

---

## Real-World Analogy First

### The Detective vs The Novelist

**Discriminative model = Detective Sherlock Holmes**
Sherlock doesn't care how a crime was committed in full detail. He only cares about *distinguishing* the guilty from the innocent. He looks at clues and draws a boundary: *"Anyone with muddy boots and ink-stained fingers is the culprit."* He's optimized purely for the decision.

**Generative model = Novelist Agatha Christie**
Agatha builds a *complete mental model* of each character — their backstory, habits, psychology. She understands what a murderer looks like so thoroughly that she could write a brand new murderer from scratch. When a new suspect appears, she compares them against her mental models.

**Applied to spam:**
- Sherlock (Logistic Regression) learns: *"If word-count > 50 AND exclamation-marks > 3, it's spam."*
- Agatha (Naive Bayes / GDA) learns: *"Spam emails have on average 80 words and 5 exclamation marks, with this spread. Legitimate emails have 30 words and 1 mark."* Then classifies by asking which description fits better.

## Concept: Two Philosophies of Classification

### Discriminative Models — Learn P(y | x) Directly

These models learn the **conditional probability** of the label given the features, or equivalently learn a direct mapping from features to labels.

| Model | What it learns |
|---|---|
| Logistic Regression | P(y=1 \| x) via sigmoid of a linear function |
| SVM | The maximum-margin decision boundary |
| Neural Networks | A complex nonlinear mapping x → y |
| Decision Trees | A sequence of feature thresholds |

**Advantages:**
- Generally higher accuracy with sufficient labeled data
- Fewer assumptions about data distribution
- More flexible — can model complex boundaries

**Limitation:** Cannot generate new data samples, cannot easily use unlabeled data.

---

### Generative Models — Learn the Joint Distribution P(x, y) = P(x | y) · P(y)

These models learn **what each class looks like**, then use **Bayes' theorem** to classify:

$$P(y \mid x) = \frac{P(x \mid y) \cdot P(y)}{P(x)}$$

| Model | What it learns |
|---|---|
| Naive Bayes | P(each feature \| y) independently |
| GDA (Gaussian Discriminant Analysis) | Gaussian P(x \| y=k) per class |
| VAE (Variational Autoencoder) | A latent space that generates realistic data |
| GAN | A generator that fools a discriminator |

**Advantages:**
- Can **generate** new synthetic samples
- Can leverage **unlabeled data** (semi-supervised learning)
- Naturally handles **missing features** via marginalization
- Excellent for **anomaly detection** — flag x where P(x) is very low

**Limitation:** Wrong distributional assumptions hurt performance badly.

---

### Gaussian Discriminant Analysis (GDA)

GDA assumes each class's features follow a **multivariate Gaussian** with a **shared covariance matrix** Σ:

$$P(x \mid y = k) = \mathcal{N}(x; \mu_k, \Sigma)$$

**Parameters to estimate from data:**
- $\phi = P(y=1)$ — fraction of spam emails  
- $\mu_0$ — mean feature vector for legitimate emails  
- $\mu_1$ — mean feature vector for spam emails  
- $\Sigma$ — shared covariance matrix (pooled across classes)

**Prediction via Bayes:**
$$\hat{y} = \arg\max_k \; P(x \mid y=k) \cdot P(y=k)$$

**Key theorem (Ng & Jordan, 2002):**  
When GDA's Gaussian assumption holds → GDA converges to optimal with **less data** than LR.  
When it doesn't hold → LR is more **robust** (makes no distributional assumptions).

**Important:** When the shared covariance assumption holds, GDA produces a **linear decision boundary** — exactly like logistic regression. With class-specific covariances (QDA), boundaries become quadratic.

In [ ]:
# Cell 1: Imports and reproducibility setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import multivariate_normal

# Reproducibility — same results every run
np.random.seed(42)

# Consistent, readable plot style
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'spam': '#e74c3c', 'legit': '#2ecc71', 'boundary': '#3498db'}

print("All libraries loaded successfully.")
print("Theme: Spam Email Detection")
print("Topic: Discriminative vs Generative Models")

In [ ]:
# Cell 2: Generate a spam dataset with approximately Gaussian features per class
# Feature 1: word_count — number of words in the email
# Feature 2: exclamation_count — number of exclamation marks
# GDA assumes each class is Gaussian — so we generate them that way

n_legit = 400   # 400 legitimate emails
n_spam  = 200   # 200 spam emails (class imbalance is realistic)

# Legitimate emails: shorter, fewer exclamations
mu_legit = np.array([40.0, 1.5])          # mean: 40 words, 1.5 exclamations
cov_legit = np.array([[100, 5], [5, 1]])   # some correlation

# Spam emails: longer promotional text, many exclamations
mu_spam = np.array([80.0, 6.0])            # mean: 80 words, 6 exclamations
cov_spam = np.array([[150, 10], [10, 2]])  # more variance (spammers vary style)

# Sample from Gaussians (this is exactly what GDA assumes)
X_legit = np.random.multivariate_normal(mu_legit, cov_legit, n_legit)
X_spam  = np.random.multivariate_normal(mu_spam, cov_spam, n_spam)

# Combine into a single dataset
X = np.vstack([X_legit, X_spam])
y = np.array([0]*n_legit + [1]*n_spam)  # 0=legit, 1=spam

# Clip to sensible ranges (no negative word counts)
X[:, 0] = np.clip(X[:, 0], 1, 300)   # word count
X[:, 1] = np.clip(X[:, 1], 0, 20)    # exclamation count

# Build a DataFrame for inspection
df = pd.DataFrame(X, columns=['word_count', 'exclamation_count'])
df['label'] = y
df['class_name'] = df['label'].map({0: 'Legitimate', 1: 'Spam'})

print(f"Dataset shape: {X.shape}")
print(f"\nClass distribution:\n{df['class_name'].value_counts()}")
print(f"\nFeature statistics by class:")
print(df.groupby('class_name')[['word_count', 'exclamation_count']].describe().round(2))

In [ ]:
# Cell 3: Visualize the Gaussian dataset
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot of raw data
ax = axes[0]
for label, name, color in [(0, 'Legitimate', COLORS['legit']), (1, 'Spam', COLORS['spam'])]:
    mask = y == label
    ax.scatter(X[mask, 0], X[mask, 1], c=color, alpha=0.5,
               label=name, edgecolors='white', linewidth=0.3, s=40)
ax.set_xlabel('Word Count', fontsize=12)
ax.set_ylabel('Exclamation Count', fontsize=12)
ax.set_title('Gaussian Spam Dataset\n(features approximately Gaussian per class)', fontsize=12)
ax.legend(fontsize=11)

# Marginal distributions — confirm Gaussianity visually
ax2 = axes[1]
for label, name, color in [(0, 'Legitimate', COLORS['legit']), (1, 'Spam', COLORS['spam'])]:
    mask = y == label
    ax2.hist(X[mask, 0], bins=25, alpha=0.6, color=color, label=f'{name} (word count)',
             density=True, edgecolor='white')
ax2.set_xlabel('Word Count', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Marginal Distribution of Word Count\n(bell-shaped → Gaussian assumption holds)', fontsize=12)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.suptitle('Figure 1: Gaussian Spam Dataset Overview', y=1.02, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# Cell 4: Fit Logistic Regression (discriminative model)
# LR learns P(y=1|x) directly — no assumptions about the distribution of x

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features — important even for LR for numerical stability
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Fit Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_sc, y_train)

lr_preds = lr_model.predict(X_test_sc)
lr_acc   = accuracy_score(y_test, lr_preds)

print("=== Logistic Regression (Discriminative) ===")
print(f"Test accuracy: {lr_acc:.4f} ({lr_acc*100:.1f}%)")
print(f"\nCoefficients: word_count={lr_model.coef_[0][0]:.3f}, "
      f"exclamation={lr_model.coef_[0][1]:.3f}")
print(f"Intercept: {lr_model.intercept_[0]:.3f}")
print("\nLR only learns the boundary — not the class distributions.")
print("It cannot tell you what a typical spam email looks like.")

In [ ]:
# Cell 5: Implement GDA from scratch
# GDA = Gaussian Discriminant Analysis
# Learns: phi (class prior), mu_0, mu_1 (class means), Sigma (shared covariance)

class GDA:
    """
    Gaussian Discriminant Analysis — a generative classifier.
    Assumes P(x | y=k) = N(mu_k, Sigma) with shared covariance.
    Uses Bayes theorem to compute P(y=k | x) for prediction.
    """
    def fit(self, X, y):
        n, d = X.shape
        classes = np.unique(y)

        # Step 1: Estimate class prior P(y=1)
        self.phi = np.mean(y == 1)  # fraction of spam emails

        # Step 2: Estimate class-conditional means
        self.mu = {c: X[y == c].mean(axis=0) for c in classes}

        # Step 3: Estimate shared covariance (pooled within-class scatter)
        # Sigma = (1/n) * sum_i (x_i - mu_{y_i})(x_i - mu_{y_i})^T
        self.Sigma = np.zeros((d, d))
        for c in classes:
            diff = X[y == c] - self.mu[c]       # shape: (n_c, d)
            self.Sigma += diff.T @ diff           # d x d scatter matrix
        self.Sigma /= n                           # normalize by total n

        self.classes = classes
        print(f"GDA fitted: phi={self.phi:.3f}, "
              f"mu_legit={self.mu[0].round(2)}, mu_spam={self.mu[1].round(2)}")
        print(f"Shared Sigma:\n{self.Sigma.round(2)}")
        return self

    def predict_proba(self, X):
        # Compute log P(x | y=k) + log P(y=k) for each class
        log_probs = []
        for c in self.classes:
            prior = self.phi if c == 1 else (1 - self.phi)
            # multivariate_normal.logpdf gives log P(x | mu_k, Sigma)
            log_likelihood = multivariate_normal.logpdf(
                X, mean=self.mu[c], cov=self.Sigma
            )
            log_probs.append(log_likelihood + np.log(prior))
        # Stack: shape (n_samples, n_classes)
        log_probs = np.stack(log_probs, axis=1)
        # Softmax to convert to probabilities
        log_probs -= log_probs.max(axis=1, keepdims=True)  # numerical stability
        probs = np.exp(log_probs)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes[np.argmax(self.predict_proba(X), axis=1)]

# Fit GDA on unscaled training data (GDA doesn't need scaling — it estimates Sigma)
gda = GDA()
gda.fit(X_train, y_train)

In [ ]:
# Cell 6: Evaluate GDA and compare with Logistic Regression
gda_preds = gda.predict(X_test)
gda_acc   = accuracy_score(y_test, gda_preds)

print("=" * 55)
print(f"{'Model':<30} {'Test Accuracy':>15}")
print("=" * 55)
print(f"{'Logistic Regression (LR)':<30} {lr_acc:>14.4f}")
print(f"{'GDA (from scratch)':<30} {gda_acc:>14.4f}")
print("=" * 55)
print()
print("Key insight: When data IS Gaussian (as here), GDA and LR")
print("produce very similar accuracy — sometimes GDA wins with")
print("less data because it uses the distributional structure.")
print()
print("GDA Classification Report:")
print(classification_report(y_test, gda_preds,
                            target_names=['Legitimate', 'Spam']))

In [ ]:
# Cell 7: Visualize decision boundaries — LR vs GDA on Gaussian data
# Both should produce nearly identical LINEAR boundaries on Gaussian data

def plot_boundary(ax, predict_fn, X, y, title, use_scaled=False, scaler=None):
    """Plot a 2D decision boundary for any classifier."""
    x_min, x_max = X[:, 0].min() - 5,  X[:, 0].max() + 5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    if use_scaled and scaler is not None:
        grid = scaler.transform(grid)
    Z = predict_fn(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn_r', levels=[-0.5, 0.5, 1.5])
    ax.contour(xx, yy, Z, levels=[0.5], colors=[COLORS['boundary']], linewidths=2.5)
    for label, name, color in [(0, 'Legitimate', COLORS['legit']), (1, 'Spam', COLORS['spam'])]:
        mask = y == label
        ax.scatter(X[mask, 0], X[mask, 1], c=color, alpha=0.6, label=name, s=25,
                   edgecolors='white', linewidth=0.3)
    ax.set_xlabel('Word Count', fontsize=10)
    ax.set_ylabel('Exclamation Count', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LR boundary — must unscale the grid for display, use scaled for prediction
plot_boundary(axes[0], lr_model.predict, X, y,
              'Logistic Regression\n(Discriminative: learns P(y|x))',
              use_scaled=True, scaler=scaler)

# GDA boundary — uses raw features directly
plot_boundary(axes[1], gda.predict, X, y,
              'GDA\n(Generative: learns P(x|y), then uses Bayes)',
              use_scaled=False)

plt.suptitle('Figure 2: Decision Boundaries — Gaussian Data\n'
             'Both produce near-identical linear boundaries when Gaussian assumption holds',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nObservation: On Gaussian data, LR and GDA boundaries are almost identical.")
print("This confirms the Ng & Jordan theoretical result.")

In [ ]:
# Cell 8: Non-Gaussian test — show LR outperforms GDA when assumption is violated
# Real email lengths are RIGHT-SKEWED (log-normal), not Gaussian
# GDA's Gaussian assumption breaks → LR is more robust

np.random.seed(42)

# Non-Gaussian data: log-normal word count, Poisson-like exclamation count
n_legit_ng, n_spam_ng = 300, 300

# Legitimate: log-normal word count (heavy right tail), few exclamations
X_legit_ng = np.column_stack([
    np.random.lognormal(mean=3.5, sigma=0.7, size=n_legit_ng),  # log-normal
    np.random.exponential(scale=1.2, size=n_legit_ng)            # exponential
])

# Spam: heavier log-normal, exponential exclamations
X_spam_ng = np.column_stack([
    np.random.lognormal(mean=4.2, sigma=0.9, size=n_spam_ng),
    np.random.exponential(scale=4.0, size=n_spam_ng)
])

X_ng = np.vstack([X_legit_ng, X_spam_ng])
y_ng = np.array([0]*n_legit_ng + [1]*n_spam_ng)
X_ng[:, 1] = np.clip(X_ng[:, 1], 0, 30)  # reasonable exclamation cap

# Train/test split for non-Gaussian data
Xng_tr, Xng_te, yng_tr, yng_te = train_test_split(
    X_ng, y_ng, test_size=0.2, random_state=42, stratify=y_ng
)

# Fit LR (scales features internally)
scaler_ng = StandardScaler()
Xng_tr_sc = scaler_ng.fit_transform(Xng_tr)
Xng_te_sc = scaler_ng.transform(Xng_te)
lr_ng = LogisticRegression(random_state=42, max_iter=1000)
lr_ng.fit(Xng_tr_sc, yng_tr)
lr_ng_acc = accuracy_score(yng_te, lr_ng.predict(Xng_te_sc))

# Fit GDA (on raw data — its Gaussian assumption will be violated)
gda_ng = GDA()
gda_ng.fit(Xng_tr, yng_tr)
gda_ng_acc = accuracy_score(yng_te, gda_ng.predict(Xng_te))

print("=== Non-Gaussian Data (Log-Normal / Exponential Features) ===")
print(f"Logistic Regression accuracy : {lr_ng_acc:.4f}")
print(f"GDA accuracy                 : {gda_ng_acc:.4f}")
gap = lr_ng_acc - gda_ng_acc
print(f"\nLR advantage: {gap:+.4f} ({gap*100:+.1f} percentage points)")
print("\nConclusion: When features are NOT Gaussian, GDA's assumptions")
print("are violated and its performance degrades. LR remains robust.")

In [ ]:
# Cell 9: Visualize the non-Gaussian data distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution of word count (log-normal — heavy right tail)
ax = axes[0]
for label, name, color in [(0, 'Legitimate', COLORS['legit']), (1, 'Spam', COLORS['spam'])]:
    ax.hist(X_ng[y_ng == label, 0], bins=40, alpha=0.6, color=color,
            label=name, density=True)
ax.set_xlabel('Word Count', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Word Count: Log-Normal Distribution\n(NOT Gaussian — GDA assumption violated!)', fontsize=11)
ax.legend()

# Distribution of exclamation count (exponential)
ax2 = axes[1]
for label, name, color in [(0, 'Legitimate', COLORS['legit']), (1, 'Spam', COLORS['spam'])]:
    ax2.hist(X_ng[y_ng == label, 1], bins=30, alpha=0.6, color=color,
             label=name, density=True)
ax2.set_xlabel('Exclamation Count', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Exclamation Count: Exponential Distribution\n(Heavy right tail — NOT Gaussian)', fontsize=11)
ax2.legend()

plt.suptitle('Figure 3: Non-Gaussian Feature Distributions\n'
             f'LR acc={lr_ng_acc:.3f} vs GDA acc={gda_ng_acc:.3f} — LR wins when assumption fails',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Generative power — sample synthetic spam emails from learned GDA
# This is something LR CANNOT do — it only knows P(y|x), not P(x|y)

print("=== Generative Sampling from GDA ===")
print("Logistic Regression: 'I can classify, but I cannot create.'")
print("GDA:                 'I understand what spam looks like. Watch me generate some.'\n")

# Sample 10 synthetic spam emails from the learned Gaussian
n_synthetic = 10
synthetic_spam = np.random.multivariate_normal(
    mean=gda.mu[1],    # mean of spam class
    cov=gda.Sigma,     # shared covariance
    size=n_synthetic
)
synthetic_spam[:, 0] = np.clip(synthetic_spam[:, 0], 1, 300)  # word count
synthetic_spam[:, 1] = np.clip(synthetic_spam[:, 1], 0, 20)   # exclamations

# Sample 10 synthetic legitimate emails
synthetic_legit = np.random.multivariate_normal(
    mean=gda.mu[0],
    cov=gda.Sigma,
    size=n_synthetic
)
synthetic_legit[:, 0] = np.clip(synthetic_legit[:, 0], 1, 300)
synthetic_legit[:, 1] = np.clip(synthetic_legit[:, 1], 0, 20)

print("10 Synthetic SPAM emails sampled from P(x | y=spam):")
df_synth_spam = pd.DataFrame(synthetic_spam.round(1),
                              columns=['word_count', 'exclamation_count'])
print(df_synth_spam.to_string(index=False))

print("\n10 Synthetic LEGITIMATE emails sampled from P(x | y=legit):")
df_synth_legit = pd.DataFrame(synthetic_legit.round(1),
                               columns=['word_count', 'exclamation_count'])
print(df_synth_legit.to_string(index=False))

print("\nThese look like realistic emails from each class — GDA learned the full distribution!")

In [ ]:
# Cell 11: Anomaly Detection with Generative Models
# Strategy: fit a Gaussian to LEGITIMATE emails only
# Emails that look nothing like legitimate OR spam get very low P(x) → anomaly

print("=== Anomaly Detection via Generative Model ===")
print("Idea: Model P(x | legitimate). Flag emails where P(x) is very low.")
print("These could be novel attack vectors not seen during training.\n")

# Fit a Gaussian to legitimate training emails only
X_legit_train = X_train[y_train == 0]
mu_legit_est  = X_legit_train.mean(axis=0)
cov_legit_est = np.cov(X_legit_train.T)

# Create a multivariate normal distribution for legitimate emails
legit_dist = multivariate_normal(mean=mu_legit_est, cov=cov_legit_est)

# Define test emails: normal legit, normal spam, and anomalous outliers
test_emails = {
    'Normal Legit Email 1'  : [38, 1.2],
    'Normal Legit Email 2'  : [45, 2.0],
    'Typical Spam Email'    : [82, 6.5],
    'Anomaly: Suspiciously Short': [2, 0.0],   # might be a tracking pixel
    'Anomaly: Extremely Long Rant': [280, 0.5], # unusual — neither spam nor legit profile
    'Anomaly: Scream Spam'  : [75, 18.0],       # crazy outlier
}

print(f"{'Email Type':<35} {'Word Ct':>8} {'Excl Ct':>8} {'log P(x|legit)':>16} {'Anomaly?':>10}")
print("-" * 82)
threshold = np.percentile(legit_dist.logpdf(X_legit_train), 5)  # 5th percentile as threshold

for name, feats in test_emails.items():
    log_p = legit_dist.logpdf(np.array(feats))
    is_anomaly = log_p < threshold
    flag = 'YES' if is_anomaly else 'no'
    print(f"{name:<35} {feats[0]:>8.1f} {feats[1]:>8.1f} {log_p:>16.3f} {flag:>10}")

print(f"\nAnomaly threshold (5th percentile of training log P): {threshold:.3f}")
print("\nA discriminative model (LR) can only say spam/not-spam.")
print("A generative model can say: 'This email doesn't look like ANYTHING we've seen.'")

In [ ]:
# Cell 12: Visualize anomaly detection — log P(x | legit) as a heatmap
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Create a grid over the feature space
x_range = np.linspace(0, 180, 200)
y_range = np.linspace(0, 18, 200)
xx, yy  = np.meshgrid(x_range, y_range)
grid    = np.c_[xx.ravel(), yy.ravel()]
log_probs = legit_dist.logpdf(grid).reshape(xx.shape)

# Heatmap of log P(x | legitimate)
ax = axes[0]
cf = ax.contourf(xx, yy, log_probs, levels=30, cmap='YlOrRd_r')
plt.colorbar(cf, ax=ax, label='log P(x | legitimate)')
# Overlay anomaly contour
ax.contour(xx, yy, log_probs, levels=[threshold], colors='blue',
           linewidths=2.5, linestyles='--')
# Add training data
ax.scatter(X_legit_train[:, 0], X_legit_train[:, 1], c='green',
           alpha=0.4, s=15, label='Legit training')
ax.set_xlabel('Word Count', fontsize=11)
ax.set_ylabel('Exclamation Count', fontsize=11)
ax.set_title('Anomaly Detection Landscape\n(Blue dashed = anomaly threshold)', fontsize=11)
ax.legend(fontsize=9)

# Mark the test emails on the plot
ax2 = axes[1]
cf2 = ax2.contourf(xx, yy, log_probs, levels=20, cmap='YlOrRd_r', alpha=0.7)
plt.colorbar(cf2, ax=ax2, label='log P(x | legitimate)')
ax2.contour(xx, yy, log_probs, levels=[threshold], colors='blue',
            linewidths=2.5, linestyles='--')
colors_map = {'no': 'green', 'YES': 'red'}
for name, feats in test_emails.items():
    log_p   = legit_dist.logpdf(np.array(feats))
    is_anom = log_p < threshold
    color   = 'red' if is_anom else 'green'
    ax2.scatter(feats[0], feats[1], c=color, s=120, zorder=5,
                edgecolors='black', linewidth=1.5)
    ax2.annotate(name.split(':')[-1].strip()[:15], xy=(feats[0], feats[1]),
                 xytext=(3, 3), textcoords='offset points', fontsize=7)
ax2.set_xlabel('Word Count', fontsize=11)
ax2.set_ylabel('Exclamation Count', fontsize=11)
ax2.set_title('Test Emails on Anomaly Map\n(Red=Anomaly, Green=Normal)', fontsize=11)

plt.suptitle('Figure 4: Generative Model for Anomaly Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Comparison Table: Discriminative vs Generative

| Aspect | Discriminative Models | Generative Models |
|---|---|---|
| **What it models** | P(y \| x) directly | P(x \| y) and P(y), then derives P(y \| x) |
| **Core question** | "Which class does x belong to?" | "Which class's description does x fit?" |
| **Examples** | Logistic Regression, SVM, Neural Nets, Decision Trees | Naive Bayes, GDA, VAE, GAN, HMM |
| **Assumptions** | Minimal (flexible boundary shape) | Strong (e.g., Gaussian per class in GDA) |
| **Accuracy (large data)** | Usually higher | Usually lower |
| **Data efficiency** | Needs more labeled data | Converges faster when assumptions hold |
| **Can generate data?** | No | Yes |
| **Handles missing features?** | Hard — needs imputation | Natural — marginalize over missing |
| **Anomaly detection** | Limited (just spam/not-spam) | Natural — flag low P(x) |
| **Unlabeled data** | Cannot use directly | Semi-supervised: use P(x) for structure |
| **When assumption wrong** | Robust | Degrades (GDA on non-Gaussian data) |
| **Use when** | Enough labeled data, accuracy is priority | Data generation, anomaly detection, less labeled data |

In [ ]:
# Cell 13: Summary visualization — learning curves comparing GDA vs LR as labeled data grows
# Ng & Jordan theorem: GDA converges faster when assumption holds

np.random.seed(42)
train_sizes = [20, 40, 80, 160, 320, 480]  # number of labeled examples
n_repeats   = 20                             # repeat each size for stability

lr_accs, gda_accs = [], []

for n_train in train_sizes:
    lr_run, gda_run = [], []
    for _ in range(n_repeats):
        # Random subsample of training data
        idx = np.random.choice(len(X_train), min(n_train, len(X_train)), replace=False)
        Xtr, ytr = X_train[idx], y_train[idx]
        if len(np.unique(ytr)) < 2:
            continue  # skip if only one class sampled
        # Fit and evaluate LR
        sc = StandardScaler().fit(Xtr)
        lr_tmp = LogisticRegression(max_iter=1000, random_state=0)
        lr_tmp.fit(sc.transform(Xtr), ytr)
        lr_run.append(accuracy_score(y_test, lr_tmp.predict(sc.transform(X_test))))
        # Fit and evaluate GDA
        gda_tmp = GDA().fit(Xtr, ytr)
        gda_run.append(accuracy_score(y_test, gda_tmp.predict(X_test)))
    lr_accs.append(np.mean(lr_run))
    gda_accs.append(np.mean(gda_run))

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, lr_accs,  'o-', color=COLORS['boundary'], label='Logistic Regression',
         linewidth=2.5, markersize=8)
plt.plot(train_sizes, gda_accs, 's--', color=COLORS['spam'],   label='GDA (generative)',
         linewidth=2.5, markersize=8)
plt.xlabel('Number of Labeled Training Examples', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Figure 5: Learning Curves — GDA vs Logistic Regression\n'
          '(Gaussian data: GDA may converge faster with fewer labels)', fontsize=12)
plt.legend(fontsize=11)
plt.ylim(0.5, 1.0)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("Key takeaway: With Gaussian data, GDA often reaches good accuracy")
print("faster (with fewer labeled examples) because it exploits the known structure.")

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** GDA assumes that features follow a Gaussian distribution per class. However, `email_length` in real datasets is right-skewed (most emails are short, a few are very long). What should you do?

<details>
<summary>Click to reveal answer</summary>

**Answer:** You have three options:

1. **Apply a log transform** to `email_length`: `log_length = log(email_length + 1)`. Log-transforming log-normal data makes it approximately Gaussian, satisfying GDA's assumption.
2. **Switch to Naive Bayes** with an appropriate distribution (e.g., `GaussianNB` still assumes Gaussian per feature, but you could use kernel density estimation or a different generative model).
3. **Switch to a discriminative model** (Logistic Regression) which makes no distributional assumptions about the features. As Ng & Jordan showed, when GDA's assumptions are violated, LR is more robust.

The key insight: GDA's Gaussian assumption is strong. Violating it hurts. Always visualize your feature distributions before choosing GDA.

</details>

---

**Question 2:** You have 10,000 unlabeled emails and only 100 labeled emails. Should you use a discriminative or generative model, and why?

<details>
<summary>Click to reveal answer</summary>

**Answer:** **Generative model**, because:

- A generative model learns `P(x)` — the overall distribution of email features — from **all** data, labeled or not.
- With only 100 labeled examples, a discriminative model (LR) has very little signal for the decision boundary and may overfit.
- A generative model uses the 10,000 unlabeled examples to build a good model of what emails look like in general. The 100 labels then inform which regions of feature space are spam vs legitimate.
- This is the basis of **semi-supervised learning**. Generative models (like GMMs, VAEs) can leverage unlabeled data through the density `P(x)`, which discriminative models cannot access.

Practical approach: Use a Gaussian Mixture Model or EM algorithm on all 10,100 emails, then use the 100 labels to assign class meanings to clusters.

</details>

---

**Question 3:** A generative model can detect emails that look like *neither* spam *nor* legitimate email — things it has never seen before. A discriminative model fundamentally cannot do this. Why?

<details>
<summary>Click to reveal answer</summary>

**Answer:** This comes down to what each model learns:

- **Discriminative model (LR):** Learns only `P(y | x)` — the probability of class given features. For any input `x`, it *must* output a class label. It has no way to say "I've never seen anything like this." It will confidently predict spam or not-spam even for a completely alien email (e.g., just a single pixel image, or an email in an ancient language).

- **Generative model (GDA, GMM):** Learns `P(x)` — the likelihood of any feature vector. For an anomalous email, `P(x)` will be near zero, because the model has no component that explains such an email. You can threshold: if `log P(x) < threshold`, declare it an anomaly rather than assigning any class.

**Spam analogy:** A new attacker might craft emails that cleverly avoid both spam and legitimate patterns (adversarial inputs). A discriminative filter says "it's not spam" and passes it through. A generative filter says "this looks like nothing in our training data — flag it for human review."

This is why generative models are the backbone of **anomaly detection** and **novelty detection** systems.

</details>

In [ ]:
# Cell 14: Final summary recap
print("=" * 60)
print(" NOTEBOOK 7 SUMMARY: Discriminative vs Generative Models")
print("=" * 60)
print()
print("DISCRIMINATIVE (Logistic Regression):")
print("  Learns P(y|x) directly")
print("  Better accuracy with lots of labeled data")
print("  No distributional assumptions")
print("  Cannot generate samples or detect anomalies")
print()
print("GENERATIVE (GDA, Naive Bayes):")
print("  Learns P(x|y) and P(y), derives P(y|x) via Bayes")
print("  Works with less labeled data (when assumptions hold)")
print("  Can generate synthetic emails")
print("  Natural anomaly detection via low P(x)")
print("  Assumption violations hurt performance")
print()
print("GDA PARAMETERS:")
print(f"  phi (spam prior)   : {gda.phi:.3f}")
print(f"  mu_legit           : {gda.mu[0].round(2)}")
print(f"  mu_spam            : {gda.mu[1].round(2)}")
print(f"  Shared Sigma       : \n{gda.Sigma.round(2)}")
print()
print("KEY THEOREM (Ng & Jordan 2002):")
print("  If Gaussian assumption holds → GDA needs less data")
print("  If assumption wrong → Logistic Regression is more robust")
print()
print("Next: Notebook 8 — k-Nearest Neighbors")

## When to Choose Which Model: Decision Guide

Use this flowchart logic before picking a classifier:

```
Do you need to GENERATE new data samples (e.g., data augmentation)?
  YES → Generative model (GDA, VAE, GAN)
  NO  ↓

Do you need ANOMALY DETECTION (flag unknown patterns)?
  YES → Generative model — use low P(x) as anomaly signal
  NO  ↓

Do you have UNLABELED data you want to leverage?
  YES → Generative model (semi-supervised via P(x))
  NO  ↓

Do your features follow known GAUSSIAN distributions per class?
  YES → GDA is a strong candidate (data-efficient)
  NO  ↓

Do you have ABUNDANT labeled data and accuracy is the priority?
  YES → Discriminative model (Logistic Regression, SVM, Neural Net)
  NO  → Generative model with careful distribution checking
```

**Practical rule of thumb:** Start with Logistic Regression (discriminative). If you need generation, anomaly detection, or have very few labels, explore generative alternatives.

In [ ]:
# Cell 15: QDA — Quadratic Discriminant Analysis (class-specific covariances)
# When we allow DIFFERENT Sigma per class, the boundary becomes quadratic
# Compare: GDA (shared Sigma → linear boundary) vs QDA (separate Sigma → curved)

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Fit sklearn LDA (equivalent to GDA with shared covariance)
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
lda_acc = accuracy_score(y_test, lda.predict(X_test))

# Fit sklearn QDA (class-specific covariances → quadratic boundary)
qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train, y_train)
qda_acc = accuracy_score(y_test, qda.predict(X_test))

print("=== LDA vs QDA: Covariance Assumption Comparison ===")
print(f"LDA (shared covariance, linear boundary)    : {lda_acc:.4f}")
print(f"QDA (per-class covariance, quadratic)       : {qda_acc:.4f}")
print(f"Our GDA from scratch                        : {gda_acc:.4f}")
print()

# Visualize both boundaries side-by-side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, clf, title in [
    (axes[0], lda, 'LDA (shared Sigma)\nLinear decision boundary'),
    (axes[1], qda, 'QDA (per-class Sigma)\nQuadratic decision boundary')
]:
    x_min, x_max = X[:, 0].min()-5, X[:, 0].max()+5
    y_min, y_max = X[:, 1].min()-0.5, X[:, 1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250),
                         np.linspace(y_min, y_max, 250))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn_r', levels=[-0.5, 0.5, 1.5])
    ax.contour(xx, yy, Z, levels=[0.5], colors=['navy'], linewidths=2.5)
    for lbl, name, color in [(0, 'Legitimate', COLORS['legit']), (1, 'Spam', COLORS['spam'])]:
        mask = y == lbl
        ax.scatter(X[mask, 0], X[mask, 1], c=color, alpha=0.5, s=25, label=name)
    ax.set_xlabel('Word Count')
    ax.set_ylabel('Exclamation Count')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
plt.suptitle('Figure 6: LDA vs QDA — Shared vs Per-Class Covariance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16: Posterior probability calibration — do LR and GDA agree?
# Both should produce sigmoidal P(spam|x) curves on Gaussian data
# Compare P(spam | x) from LR vs GDA along a 1D slice of feature space

# Create a 1D slice: fix exclamation_count=3, vary word_count from 10 to 130
word_counts = np.linspace(10, 130, 200)
excl_fixed  = 3.0   # hold exclamation count constant
X_slice_raw = np.column_stack([word_counts, np.full(200, excl_fixed)])
X_slice_sc  = scaler.transform(X_slice_raw)

# LR probability: sigmoid of a linear score
lr_probs  = lr_model.predict_proba(X_slice_sc)[:, 1]

# GDA probability: derived from Bayes theorem applied to Gaussian likelihoods
gda_probs = gda.predict_proba(X_slice_raw)[:, 1]

# Plot the posterior probability curves side-by-side
plt.figure(figsize=(10, 5))
plt.plot(word_counts, lr_probs,  color=COLORS['boundary'], linewidth=2.5,
         label='Logistic Regression P(spam|x)')
plt.plot(word_counts, gda_probs, color=COLORS['spam'], linewidth=2.5,
         linestyle='--', label='GDA P(spam|x) via Bayes')
plt.axhline(0.5, color='gray', linestyle=':', linewidth=1.5, label='Decision threshold (0.5)')
plt.fill_between(word_counts, 0, 0.5, alpha=0.06, color=COLORS['legit'], label='Predicted: Legit')
plt.fill_between(word_counts, 0.5, 1.0, alpha=0.06, color=COLORS['spam'], label='Predicted: Spam')
plt.xlabel('Word Count (exclamation_count fixed at 3)', fontsize=12)
plt.ylabel('P(spam | x)', fontsize=12)
plt.title('Figure 7: Posterior Probability — LR vs GDA Along a 1D Slice\n'
          'Both curves are sigmoidal on Gaussian data (theory confirmed)', fontsize=11)
plt.legend(fontsize=10)
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# Find the word_count where each model crosses 50% (the decision boundary)
lr_cross  = word_counts[np.argmin(np.abs(lr_probs - 0.5))]
gda_cross = word_counts[np.argmin(np.abs(gda_probs - 0.5))]
print(f"LR decision boundary at word_count  = {lr_cross:.1f}  (excl=3)")
print(f"GDA decision boundary at word_count = {gda_cross:.1f}  (excl=3)")
print("\nBoth models cross at nearly the same word count — confirming their")
print("near-identical decision boundaries on Gaussian data.")

In [ ]:
# Cell 17: Seaborn pairplot — visually verify the Gaussian assumption
# Each subplot shows the bivariate relationship between two features
# Diagonal KDEs confirm (or deny) that features look Gaussian per class
# This is a FIRST STEP before deciding to use GDA

np.random.seed(42)
n_legit_pp, n_spam_pp = 250, 150

# Three-feature dataset for a richer pairplot
X_legit_pp = np.column_stack([
    np.random.normal(40, 10, n_legit_pp),    # word_count
    np.random.normal(1.2, 0.7, n_legit_pp),  # exclamation_count
    np.random.normal(1.5, 0.9, n_legit_pp),  # link_count
])
X_spam_pp = np.column_stack([
    np.random.normal(80, 20, n_spam_pp),
    np.random.normal(6.0, 2.0, n_spam_pp),
    np.random.normal(5.5, 1.8, n_spam_pp),
])
X_pp = np.clip(np.vstack([X_legit_pp, X_spam_pp]), 0, None)
y_pp = ['Legitimate'] * n_legit_pp + ['Spam'] * n_spam_pp

df_pp = pd.DataFrame(X_pp, columns=['word_count', 'exclamation_count', 'link_count'])
df_pp['Class'] = y_pp

# Pairplot: diagonal=KDE (check bell shape), off-diagonal=scatter (check elliptical)
g = sns.pairplot(
    df_pp,
    hue='Class',
    palette={'Legitimate': COLORS['legit'], 'Spam': COLORS['spam']},
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 's': 25},
    diag_kws={'linewidth': 2.5}
)
g.fig.suptitle('Figure 8: Pairplot — Verifying Gaussian Assumption Per Class\n'
               'Bell-shaped KDE diagonals = assumption reasonable; '
               'elliptical scatter = shared covariance plausible',
               y=1.02, fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("How to read this plot before using GDA:")
print("  Diagonal KDEs: should look bell-shaped (Gaussian) for each class separately")
print("  Off-diagonal scatter: clusters should be roughly elliptical, not crescent-shaped")
print("  If either check fails → apply log transform or switch to LR")